## LBG Training Dataset Construction

This notebook builds a training dataset for LBG (Lyman-Break Galaxy) classification
by matching DESI spectroscopic data with CLAUDS photometric data.

## Workflow
1. Load and inspect data
2. Coordinate matching (Specz → Raw photometry)
3. Apply quality selection
4. Determine LBG labels from spectroscopic redshifts
5. Extract photometric features with proper non-detection handling
6. Build final training dataset

In [1]:
# Cell 1: Imports and Configuration
import numpy as np
import pandas as pd
from astropy.table import Table
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy import units as u
from typing import Dict, List, Tuple
import warnings
import os

warnings.filterwarnings('ignore')

# Create output directory if not exists
os.makedirs("../data_processed", exist_ok=True)

print("Imports completed successfully.")

Imports completed successfully.


In [2]:
# Cell 2: Configuration Parameters

# =============================================================================
# CONFIGURATION
# =============================================================================

# --- File Paths ---
CONFIGS = [
    {
        'path_specz': "../CLAUDS_udrop_specz/COSMOS_udrop_specz.fits",
        'path_raw': "../data_Clauds/COSMOS_6bands-SExtractor-Lephare.fits",
        'field_name': "COSMOS"
    },
    {
        'path_specz': "../CLAUDS_udrop_specz/XMM_udrop_specz.fits",
        'path_raw': "../data_Clauds/XMMLSS_6bands-SExtractor-Lephare.fits",
        'field_name': "XMM-LSS"
    }
]

# --- Matching Parameters ---
MATCH_RADIUS_ARCSEC = 1.0

# --- LBG Selection Criteria ---
Z_THRESHOLD = 2.0           # Redshift threshold for LBG classification
VI_QUALITY_MIN = 2.5        # Minimum VI quality for reliable redshift
RR_DELTACHI2_MIN = 9.0      # Minimum RR delta chi2 for reliable redshift

# --- Photometry Configuration ---
# Column mapping:  internal name -> catalog column name
PHOTOMETRY_COLUMNS = {
    # Aperture magnitudes (2 arcsec)
    'mag_u': 'MAG_APER_2s_uS',
    'mag_g': 'MAG_APER_2s_g',
    'mag_r': 'MAG_APER_2s_r',
    'mag_i': 'MAG_APER_2s_i',
    'mag_z': 'MAG_APER_2s_z',
    'mag_y': 'MAG_APER_2s_y',
    # Magnitude errors
    'err_u': 'MAGERR_APER_2s_uS',
    'err_g': 'MAGERR_APER_2s_g',
    'err_r': 'MAGERR_APER_2s_r',
    'err_i': 'MAGERR_APER_2s_i',
    'err_z': 'MAGERR_APER_2s_z',
    'err_y': 'MAGERR_APER_2s_y',
    # Magnitude offset (zero-point correction)
    'offset_mag': 'OFFSET_MAG_2s',
}

# Limiting magnitudes for non-detection filling (5-sigma depth)
LIMITING_MAGS = {
    'u': 27.7,
    'g': 27.4,
    'r': 27.1,
    'i': 26.9,
    'z': 26.3,
    'y': 26.0,
}

# SNR threshold for reliable detection
SNR_THRESHOLD = 5.0

# Conversion factor:  SNR = 1.0857 / mag_err (where 1.0857 = 2.5/ln(10))
MAG_ERR_TO_SNR_FACTOR = 1.0857

# Color definitions:  (color_name, band1, band2) -> color = band1 - band2
COLORS_TO_COMPUTE = [
    ('u_g', 'mag_u', 'mag_g'),
    ('g_r', 'mag_g', 'mag_r'),
    ('r_i', 'mag_r', 'mag_i'),
    ('i_z', 'mag_i', 'mag_z'),
    ('z_y', 'mag_z', 'mag_y'),
]

print("Configuration loaded.")
print(f"  Match radius: {MATCH_RADIUS_ARCSEC} arcsec")
print(f"  LBG z threshold: {Z_THRESHOLD}")
print(f"  SNR threshold: {SNR_THRESHOLD}")

Configuration loaded.
  Match radius: 1.0 arcsec
  LBG z threshold: 2.0
  SNR threshold: 5.0


In [3]:
# 3: Helper Functions - Data Quality

def compute_snr_from_mag_err(mag_err: np.ndarray) -> np.ndarray:
    """
    Compute Signal-to-Noise Ratio from magnitude error. 
    
    SNR = 1.0857 / mag_err, where 1.0857 = 2.5 / ln(10)
    
    Parameters
    ----------
    mag_err : np.ndarray
        Magnitude error array
        
    Returns
    -------
    snr : np.ndarray
        Signal-to-noise ratio (NaN for invalid mag_err)
    """
    # Handle invalid values
    valid = (mag_err > 0) & (mag_err < 99) & np.isfinite(mag_err)
    snr = np.full_like(mag_err, np.nan, dtype=float)
    snr[valid] = MAG_ERR_TO_SNR_FACTOR / mag_err[valid]
    return snr


def is_reliable_detection(mag_err: np.ndarray, snr_threshold: float = SNR_THRESHOLD) -> np.ndarray:
    """
    Determine if a measurement is a reliable detection based on SNR.
    
    Parameters
    ----------
    mag_err : np.ndarray
        Magnitude error array
    snr_threshold : float
        Minimum SNR for reliable detection (default: 5.0)
        
    Returns
    -------
    is_detected : np.ndarray
        Boolean array (True = reliable detection)
    """
    snr = compute_snr_from_mag_err(mag_err)
    return snr >= snr_threshold


def process_magnitude_with_snr(
    mag:  np.ndarray,
    mag_err: np.ndarray,
    band: str,
    snr_threshold: float = SNR_THRESHOLD
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Process magnitude values with SNR-based detection criterion.
    
    For non-detections (SNR < threshold), magnitude is set to limiting magnitude.
    
    Parameters
    ----------
    mag : np.ndarray
        Raw magnitude values
    mag_err : np. ndarray
        Magnitude errors
    band : str
        Band name (u, g, r, i, z, y) for limiting magnitude lookup
    snr_threshold : float
        SNR threshold for reliable detection
        
    Returns
    -------
    mag_processed : np.ndarray
        Processed magnitudes (non-detections filled with limiting mag)
    is_detected : np.ndarray
        Detection flag (1 = detected, 0 = non-detection)
    snr :  np.ndarray
        Computed SNR values
    """
    # Get limiting magnitude for this band
    limit_mag = LIMITING_MAGS.get(band, 27.7)
    
    # Compute SNR and detection flag
    snr = compute_snr_from_mag_err(mag_err)
    is_detected = (snr >= snr_threshold).astype(int)
    
    # Also check for obviously invalid magnitude values
    is_valid_mag = np.isfinite(mag) & (mag > 0) & (mag < 99)
    is_detected = is_detected & is_valid_mag.astype(int)
    
    # Fill non-detections with limiting magnitude
    mag_processed = np.where(is_detected == 1, mag, limit_mag)
    
    return mag_processed, is_detected, snr


print("Helper functions (Data Quality) defined.")

Helper functions (Data Quality) defined.


In [4]:
# Cell 4: Helper Functions - Selection Criteria

def apply_field_selection(table: Table, field: str) -> np.ndarray:
    """
    Apply recommended quality selection criteria.
    
    Selection criteria:
    - COSMOS: (MASK == 0) & FLAG_FIELD_BINARY[:,0] & FLAG_FIELD_BINARY[:,1]
    - XMMLSS: (MASK == 0) & FLAG_FIELD_BINARY[:,0] & FLAG_FIELD_BINARY[:,2]
    
    Parameters
    ----------
    table : astropy.table.Table
        Input photometric catalog
    field : str
        Field name ('COSMOS' or 'XMM-LSS')
        
    Returns
    -------
    selection_mask : np.ndarray
        Boolean mask for selected sources
    """
    mask = np.array(table['MASK'])
    flag_field = np.array(table['FLAG_FIELD_BINARY'])
    
    # Condition 1: Good pixel mask
    cond_mask = (mask == 0)
    
    # Condition 2: In valid observation region
    cond_flag0 = flag_field[:, 0].astype(bool)
    
    # Condition 3: Field-specific flag
    if field. upper() == 'COSMOS':
        cond_field = flag_field[:, 1].astype(bool)
    else:  # XMM-LSS
        cond_field = flag_field[:, 2].astype(bool)
    
    return cond_mask & cond_flag0 & cond_field


def select_lbg_from_specz(
    table: Table,
    z_threshold: float = Z_THRESHOLD,
    vi_quality_min: float = VI_QUALITY_MIN,
    rr_deltachi2_min: float = RR_DELTACHI2_MIN
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Determine LBG labels from spectroscopic data.
    
    Reliability criteria:
    - VI:  VI_QUALITY >= 2.5
    - RR:  RR_DELTACHI2 > 9.0
    
    LBG definition:  z > z_threshold (default 2.0)
    
    Parameters
    ----------
    table :  astropy.table.Table
        Spectroscopic catalog
    z_threshold : float
        Redshift threshold for LBG classification
    vi_quality_min : float
        Minimum VI quality for reliable redshift
    rr_deltachi2_min : float
        Minimum RR delta chi2 for reliable redshift
        
    Returns
    -------
    z_best : np.ndarray
        Best redshift estimate (VI preferred over RR)
    is_lbg : np.ndarray
        LBG label (1 = LBG, 0 = non-LBG, -1 = unreliable)
    is_reliable : np.ndarray
        Reliability flag (True = reliable redshift)
    """
    # Extract columns
    vi_z = np.nan_to_num(np.array(table['VI_Z']), nan=-99.0)
    vi_quality = np.nan_to_num(np.array(table['VI_QUALITY']), nan=-99.0)
    rr_z = np.nan_to_num(np.array(table['RR_Z']), nan=-99.0)
    rr_deltachi2 = np.nan_to_num(np.array(table['RR_DELTACHI2']), nan=-99.0)
    
    # Determine reliability
    is_vi_reliable = (vi_quality >= vi_quality_min)
    is_rr_reliable = (rr_deltachi2 > rr_deltachi2_min)
    is_reliable = is_vi_reliable | is_rr_reliable
    
    # Best redshift: prefer VI over RR
    z_best = np.where(is_vi_reliable & (vi_z > 0), vi_z, rr_z)
    
    # LBG classification
    # -1: unreliable, 0: non-LBG (z <= threshold), 1: LBG (z > threshold)
    is_lbg = np.full(len(table), -1, dtype=int)
    is_lbg[is_reliable & (z_best > z_threshold)] = 1
    is_lbg[is_reliable & (z_best <= z_threshold) & (z_best > 0)] = 0
    
    return z_best, is_lbg, is_reliable


print("Helper functions (Selection Criteria) defined.")

Helper functions (Selection Criteria) defined.


In [13]:
# Cell 5: Main Processing Function

def process_single_field(
    path_specz: str,
    path_raw: str,
    field_name: str,
    match_radius: float = MATCH_RADIUS_ARCSEC,
    require_reliable_z: bool = True,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Process a single field: match, select, and extract features.
    
    Parameters
    ----------
    path_specz : str
        Path to DESI spectroscopic catalog
    path_raw : str
        Path to CLAUDS photometric catalog
    field_name : str
        Field identifier ('COSMOS' or 'XMM-LSS')
    match_radius : float
        Coordinate matching radius in arcsec
    require_reliable_z : bool
        If True, only keep sources with reliable redshifts
    verbose : bool
        Print progress information
        
    Returns
    -------
    df : pd.DataFrame
        Processed data with features and labels
    """
    
    if verbose:
        print("=" * 70)
        print(f"Processing: {field_name}")
        print("=" * 70)
    
    # =========================================================================
    # Step 1: Load data
    # =========================================================================
    if verbose:
        print("\n[1] Loading data...")
    
    tab_specz = Table.read(path_specz, hdu=1)
    tab_raw = Table.read(path_raw, hdu=1, memmap=True)
    
    n_specz, n_raw = len(tab_specz), len(tab_raw)
    if verbose:
        print(f"    Specz catalog: {n_specz:,} sources")
        print(f"    Raw catalog: {n_raw:,} sources")
    
    # =========================================================================
    # Step 2: Coordinate matching
    # =========================================================================
    if verbose: 
        print("\n[2] Coordinate matching...")
    
    # Determine coordinate column names
    ra_col = 'TARGET_RA' if 'TARGET_RA' in tab_specz.colnames else 'RA'
    dec_col = 'TARGET_DEC' if 'TARGET_DEC' in tab_specz.colnames else 'DEC'
    
    coords_specz = SkyCoord(
        ra=tab_specz[ra_col] * u.deg,
        dec=tab_specz[dec_col] * u.deg
    )
    coords_raw = SkyCoord(
        ra=tab_raw['RA'] * u.deg,
        dec=tab_raw['DEC'] * u.deg
    )
    
    # Find nearest neighbor for each specz source
    idx_raw, d2d, _ = match_coordinates_sky(coords_specz, coords_raw)
    separations = d2d.to(u.arcsec).value
    
    # Apply matching radius cut
    good_match = separations < match_radius
    n_matched = np.sum(good_match)
    
    if verbose: 
        print(f"    Matched (< {match_radius}\"): {n_matched:,} ({n_matched/n_specz*100:.1f}%)")
    
    # Get indices of matched sources
    matched_specz_idx = np.where(good_match)[0]
    matched_raw_idx = idx_raw[good_match]
    
    # =========================================================================
    # Step 3: Apply field selection on matched sources
    # =========================================================================
    if verbose:
        print("\n[3] Applying field quality selection...")
    
    # Extract matched raw sources
    tab_raw_matched = tab_raw[matched_raw_idx]
    field_mask = apply_field_selection(tab_raw_matched, field_name)
    
    n_pass = np.sum(field_mask)
    if verbose:
        print(f"    Pass selection: {n_pass:,} ({n_pass/n_matched*100:.1f}%)")
    
    # Update indices to only include sources passing selection
    final_specz_idx = matched_specz_idx[field_mask]
    final_raw_idx = matched_raw_idx[field_mask]
    
    # =========================================================================
    # Step 4: Determine LBG labels
    # =========================================================================
    if verbose: 
        print("\n[4] Determining LBG labels...")
    
    tab_specz_final = tab_specz[final_specz_idx]
    z_best, is_lbg, is_reliable = select_lbg_from_specz(tab_specz_final)
    
    n_reliable = np.sum(is_reliable)
    n_lbg = np.sum(is_lbg == 1)
    n_non_lbg = np.sum(is_lbg == 0)
    
    if verbose:
        print(f"    Reliable z: {n_reliable:,} ({n_reliable/len(is_lbg)*100:.1f}%)")
        print(f"    LBG (z > {Z_THRESHOLD}): {n_lbg:,}")
        print(f"    Non-LBG:  {n_non_lbg:,}")
    
    # Filter to reliable redshifts only
    if require_reliable_z:
        reliable_mask = is_reliable
        final_specz_idx = final_specz_idx[reliable_mask]
        final_raw_idx = final_raw_idx[reliable_mask]
        z_best = z_best[reliable_mask]
        is_lbg = is_lbg[reliable_mask]
        
        if verbose:
            print(f"    After reliability cut: {len(final_specz_idx):,}")
    
    # =========================================================================
    # Step 5: Extract photometric features
    # =========================================================================
    if verbose:
        print("\n[5] Extracting photometric features...")
    
    tab_raw_final = tab_raw[final_raw_idx]
    n_final = len(tab_raw_final)
    
    # Initialize DataFrame
    data_dict = {}
    
    # --- Basic info ---
    # Use list/array with explicit length to avoid index issues
    data_dict['field'] = [field_name] * n_final  # List of strings
    data_dict['specz_idx'] = final_specz_idx.astype(int)
    data_dict['raw_idx'] = final_raw_idx.astype(int)
    data_dict['RA'] = np.array(tab_raw_final['RA'])
    data_dict['DEC'] = np. array(tab_raw_final['DEC'])

    # --- Labels ---
    data_dict['z_best'] = z_best
    data_dict['is_lbg'] = is_lbg

    # --- Magnitude offset ---
    if 'OFFSET_MAG_2s' in tab_raw_final.colnames:
        offset_mag = np.array(tab_raw_final['OFFSET_MAG_2s'])
        data_dict['offset_mag'] = offset_mag
        if verbose:
            valid_offset = offset_mag[np.isfinite(offset_mag)]
            if len(valid_offset) > 0:
                print(f"    OFFSET_MAG_2s:  mean={np.mean(valid_offset):.4f}, std={np.std(valid_offset):.4f}")
    else:
        offset_mag = np.zeros(n_final)
        data_dict['offset_mag'] = offset_mag
    
    # --- Extract raw magnitudes and errors ---
    bands = ['u', 'g', 'r', 'i', 'z', 'y']
    
    for band in bands:
        mag_col = PHOTOMETRY_COLUMNS.get(f'mag_{band}')
        err_col = PHOTOMETRY_COLUMNS.get(f'err_{band}')
        
        if mag_col in tab_raw_final.colnames and err_col in tab_raw_final.colnames:
            mag_raw = np.array(tab_raw_final[mag_col])
            mag_err = np.array(tab_raw_final[err_col])
            
            # Apply magnitude offset correction
            # Note: Check if offset should be added or subtracted based on catalog convention
            # Typically:  mag_corrected = mag_raw + offset
            mag_corrected = mag_raw + offset_mag
            
            # Process with SNR criterion
            mag_processed, is_detected, snr = process_magnitude_with_snr(
                mag_corrected, mag_err, band
            )
            
            # Store in data dict
            data_dict[f'mag_{band}_raw'] = mag_raw
            data_dict[f'mag_{band}'] = mag_processed
            data_dict[f'err_{band}'] = mag_err
            data_dict[f'snr_{band}'] = snr
            data_dict[f'flag_{band}'] = is_detected
        else:
            if verbose:
                print(f"    Warning: {mag_col} or {err_col} not found")
    
    # --- Additional columns ---
    if 'CLASS_STAR_HSC_I' in tab_raw_final.colnames:
        class_star = np.array(tab_raw_final['CLASS_STAR_HSC_I'])
        class_star = np.where(
            (class_star >= 0) & (class_star <= 1),
            class_star, np.nan
        )
        data_dict['class_star'] = class_star
    
    if 'OBJ_TYPE' in tab_raw_final.colnames:
        data_dict['obj_type'] = np.array(tab_raw_final['OBJ_TYPE'])
    
    if 'Z_BEST' in tab_raw_final.colnames:
        data_dict['photo_z'] = np.array(tab_raw_final['Z_BEST'])

    # -------------------------------------------------------------------------
    # Create DataFrame from dictionary
    # -------------------------------------------------------------------------
    df = pd.DataFrame(data_dict)
    
    # --- Compute colors ---
    for color_name, band1, band2 in COLORS_TO_COMPUTE:
        if band1 in df.columns and band2 in df.columns:
            df[color_name] = df[band1] - df[band2]
    
    if verbose:
        print(f"\n    Final DataFrame:  {df.shape[0]} rows, {df.shape[1]} columns")
        # Verify field column
        print(f"    Field column dtype: {df['field'].dtype}, unique values: {df['field'].unique()}")
    
    return df


print("Main processing function defined.")

Main processing function defined.


In [14]:
# Cell 6: Process Both Fields

# Process each field
dfs = []

for config in CONFIGS:
    df_field = process_single_field(
        path_specz=config['path_specz'],
        path_raw=config['path_raw'],
        field_name=config['field_name'],
        match_radius=MATCH_RADIUS_ARCSEC,
        require_reliable_z=True,
        verbose=True
    )
    dfs.append(df_field)
    print()

# Combine all fields
df_train = pd.concat(dfs, ignore_index=True)

print("=" * 70)
print("COMBINED DATASET")
print("=" * 70)
print(f"Total samples: {len(df_train):,}")

Processing: COSMOS

[1] Loading data...
    Specz catalog: 4,486 sources
    Raw catalog: 5,263,013 sources

[2] Coordinate matching...
    Matched (< 1.0"): 4,486 (100.0%)

[3] Applying field quality selection...
    Pass selection: 4,421 (98.6%)

[4] Determining LBG labels...
    Reliable z: 3,301 (74.7%)
    LBG (z > 2.0): 1,667
    Non-LBG:  1,627
    After reliability cut: 3,301

[5] Extracting photometric features...
    OFFSET_MAG_2s:  mean=-0.4856, std=0.5382

    Final DataFrame:  3301 rows, 46 columns
    Field column dtype: object, unique values: ['COSMOS']

Processing: XMM-LSS

[1] Loading data...
    Specz catalog: 6,386 sources
    Raw catalog: 5,166,244 sources

[2] Coordinate matching...
    Matched (< 1.0"): 6,386 (100.0%)

[3] Applying field quality selection...
    Pass selection: 6,386 (100.0%)

[4] Determining LBG labels...
    Reliable z: 2,119 (33.2%)
    LBG (z > 2.0): 1,356
    Non-LBG:  750
    After reliability cut: 2,119

[5] Extracting photometric features.

In [15]:
# Cell 7: Dataset Summary and Statistics

print("=" * 70)
print("TRAINING DATASET SUMMARY")
print("=" * 70)

# --- Class distribution ---
print("\n[1] Class Distribution:")
print("-" * 40)
n_lbg = (df_train['is_lbg'] == 1).sum()
n_non_lbg = (df_train['is_lbg'] == 0).sum()
print(f"    LBG (positive):     {n_lbg:,} ({n_lbg/len(df_train)*100:.1f}%)")
print(f"    Non-LBG (negative): {n_non_lbg:,} ({n_non_lbg/len(df_train)*100:.1f}%)")
print(f"    Class ratio:        {n_lbg/n_non_lbg:.2f}:1")

# --- By field ---
print("\n[2] By Field:")
print("-" * 40)
for field in df_train['field'].unique():
    mask = df_train['field'] == field
    n_total = mask.sum()
    n_pos = (df_train.loc[mask, 'is_lbg'] == 1).sum()
    n_neg = (df_train.loc[mask, 'is_lbg'] == 0).sum()
    print(f"    {field}:")
    print(f"      Total: {n_total:,}, LBG: {n_pos:,} ({n_pos/n_total*100:.1f}%), Non-LBG: {n_neg:,}")

# --- Detection rates ---
print("\n[3] Detection Rates (SNR >= 5):")
print("-" * 40)
bands = ['u', 'g', 'r', 'i', 'z', 'y']
for band in bands:
    flag_col = f'flag_{band}'
    if flag_col in df_train.columns:
        det_rate = df_train[flag_col].mean() * 100
        print(f"    {band}-band: {det_rate:.1f}% detected")

# --- Feature statistics ---
print("\n[4] Color Statistics:")
print("-" * 40)
color_cols = ['u_g', 'g_r', 'r_i', 'i_z', 'z_y']
for col in color_cols:
    if col in df_train.columns:
        mean_val = df_train[col].mean()
        std_val = df_train[col].std()
        print(f"    {col}: mean={mean_val:.3f}, std={std_val:.3f}")

# --- LBG vs Non-LBG color comparison ---
print("\n[5] Color Comparison (LBG vs Non-LBG):")
print("-" * 40)
for col in ['u_g', 'g_r']: 
    if col in df_train.columns:
        lbg_mean = df_train.loc[df_train['is_lbg']==1, col].mean()
        non_lbg_mean = df_train.loc[df_train['is_lbg']==0, col].mean()
        print(f"    {col}: LBG={lbg_mean:.3f}, Non-LBG={non_lbg_mean:.3f}, diff={lbg_mean-non_lbg_mean:.3f}")

TRAINING DATASET SUMMARY

[1] Class Distribution:
----------------------------------------
    LBG (positive):     3,023 (55.8%)
    Non-LBG (negative): 2,377 (43.9%)
    Class ratio:        1.27:1

[2] By Field:
----------------------------------------
    COSMOS:
      Total: 3,301, LBG: 1,667 (50.5%), Non-LBG: 1,627
    XMM-LSS:
      Total: 2,119, LBG: 1,356 (64.0%), Non-LBG: 750

[3] Detection Rates (SNR >= 5):
----------------------------------------
    u-band: 73.0% detected
    g-band: 99.9% detected
    r-band: 99.9% detected
    i-band: 99.4% detected
    z-band: 99.4% detected
    y-band: 85.9% detected

[4] Color Statistics:
----------------------------------------
    u_g: mean=1.698, std=1.030
    g_r: mean=0.425, std=0.325
    r_i: mean=0.124, std=0.301
    i_z: mean=0.097, std=0.303
    z_y: mean=-0.188, std=0.688

[5] Color Comparison (LBG vs Non-LBG):
----------------------------------------
    u_g: LBG=1.732, Non-LBG=1.656, diff=0.075
    g_r: LBG=0.419, Non-LBG=0.

In [16]:
# Cell 8: Data Quality Checks

# %%
print("=" * 70)
print("DATA QUALITY CHECKS")
print("=" * 70)

# --- Check for missing values ---
print("\n[1] Missing Values:")
print("-" * 40)
feature_cols = [c for c in df_train.columns if c.startswith(('mag_', 'err_', 'snr_', 'flag_', 'u_', 'g_', 'r_', 'i_', 'z_'))]
for col in feature_cols[:10]:  # Show first 10
    n_missing = df_train[col].isna().sum()
    if n_missing > 0:
        print(f"    {col}: {n_missing} missing ({n_missing/len(df_train)*100:.2f}%)")

# --- Check offset_mag distribution ---
print("\n[2] Magnitude Offset (OFFSET_MAG_2s):")
print("-" * 40)
if 'offset_mag' in df_train.columns:
    offset = df_train['offset_mag']
    print(f"    Min:    {offset.min():.4f}")
    print(f"    Max:    {offset.max():.4f}")
    print(f"    Mean:   {offset.mean():.4f}")
    print(f"    Median: {offset.median():.4f}")

# --- Redshift distribution ---
print("\n[3] Redshift Distribution:")
print("-" * 40)
z = df_train['z_best']
print(f"    Min:    {z.min():.3f}")
print(f"    Max:    {z.max():.3f}")
print(f"    Mean:   {z.mean():.3f}")
print(f"    Median: {z.median():.3f}")

# Redshift bins
z_bins = [0, 1, 2, 3, 4, 5, 7]
print("\n    Redshift histogram:")
for i in range(len(z_bins)-1):
    n_in_bin = ((z >= z_bins[i]) & (z < z_bins[i+1])).sum()
    print(f"      z=[{z_bins[i]}, {z_bins[i+1]}): {n_in_bin:,}")

DATA QUALITY CHECKS

[1] Missing Values:
----------------------------------------
    snr_u: 582 missing (10.74%)

[2] Magnitude Offset (OFFSET_MAG_2s):
----------------------------------------
    Min:    -3.9439
    Max:    0.0000
    Mean:   -0.4252
    Median: -0.2788

[3] Redshift Distribution:
----------------------------------------
    Min:    -0.004
    Max:    5.287
    Mean:   1.915
    Median: 2.472

    Redshift histogram:
      z=[0, 1): 1,557
      z=[1, 2): 820
      z=[2, 3): 1,716
      z=[3, 4): 1,303
      z=[4, 5): 2
      z=[5, 7): 2


In [17]:
# Cell 9: Preview Sample Data

print("=" * 70)
print("SAMPLE DATA PREVIEW")
print("=" * 70)

# Select columns to display
preview_cols = [
    'field', 'RA', 'DEC', 'z_best', 'is_lbg',
    'mag_u', 'mag_g', 'mag_i', 'u_g', 'g_r',
    'flag_u', 'flag_g', 'snr_u', 'snr_g'
]
preview_cols = [c for c in preview_cols if c in df_train.columns]

print("\nFirst 10 rows:")
print(df_train[preview_cols].head(10).to_string())

print("\n\nLBG samples (first 5):")
print(df_train[df_train['is_lbg']==1][preview_cols].head().to_string())

print("\n\nNon-LBG samples (first 5):")
print(df_train[df_train['is_lbg']==0][preview_cols].head().to_string())

SAMPLE DATA PREVIEW

First 10 rows:
    field          RA       DEC    z_best  is_lbg      mag_u      mag_g      mag_i       u_g       g_r  flag_u  flag_g      snr_u      snr_g
0  COSMOS  150.386174  1.018215  0.000848       0  25.314104  24.497484  24.231743  0.816620  0.185917       1       1   9.492054  54.190739
1  COSMOS  150.337356  1.026364  0.297918       0  27.700000  25.434265  23.817093  2.265735  1.079193       0       1   0.525405  36.469727
2  COSMOS  150.408661  1.057914  0.000848       0  27.700000  24.152698  23.788873  3.547302  0.147650       0       1   1.372539  62.250603
3  COSMOS  150.439663  0.976806  0.000005       0  27.700000  24.706857  22.187066  2.993143  1.214067       0       1   1.459956  51.597126
4  COSMOS  150.356040  0.958010  1.014920       0  25.312200  24.432407  23.930747  0.879793  0.260422       1       1  10.408490  42.759880
5  COSMOS  150.423118  0.997719  0.000848       0  25.946995  25.030402  24.239579  0.916594  0.549349       1       1

In [18]:
# Cell 10: Save Dataset

# Define output path
output_path = "../data_processed/training_dataset.csv"

# Select columns to save
# Core columns
core_cols = ['field', 'specz_idx', 'raw_idx', 'RA', 'DEC', 'z_best', 'is_lbg']

# Feature columns
feature_cols = []
for band in ['u', 'g', 'r', 'i', 'z', 'y']: 
    feature_cols.extend([f'mag_{band}', f'err_{band}', f'snr_{band}', f'flag_{band}'])

# Color columns
color_cols = ['u_g', 'g_r', 'r_i', 'i_z', 'z_y']

# Additional columns
extra_cols = ['offset_mag', 'class_star', 'obj_type', 'photo_z']

# Combine all columns
all_cols = core_cols + feature_cols + color_cols + extra_cols
save_cols = [c for c in all_cols if c in df_train.columns]

# Save
df_save = df_train[save_cols]
df_save.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")
print(f"Shape: {df_save.shape}")
print(f"Columns: {list(df_save.columns)}")

Dataset saved to: ../data_processed/training_dataset.csv
Shape: (5420, 40)
Columns: ['field', 'specz_idx', 'raw_idx', 'RA', 'DEC', 'z_best', 'is_lbg', 'mag_u', 'err_u', 'snr_u', 'flag_u', 'mag_g', 'err_g', 'snr_g', 'flag_g', 'mag_r', 'err_r', 'snr_r', 'flag_r', 'mag_i', 'err_i', 'snr_i', 'flag_i', 'mag_z', 'err_z', 'snr_z', 'flag_z', 'mag_y', 'err_y', 'snr_y', 'flag_y', 'u_g', 'g_r', 'r_i', 'i_z', 'z_y', 'offset_mag', 'class_star', 'obj_type', 'photo_z']


In [19]:
# Cell 11: Quick Verification

# Reload and verify
df_verify = pd.read_csv(output_path)

print("=" * 70)
print("VERIFICATION")
print("=" * 70)
print(f"Loaded shape: {df_verify.shape}")
print(f"LBG count: {(df_verify['is_lbg']==1).sum()}")
print(f"Non-LBG count: {(df_verify['is_lbg']==0).sum()}")
print("\nData types:")
print(df_verify.dtypes)

VERIFICATION
Loaded shape: (5420, 40)
LBG count: 3023
Non-LBG count: 2377

Data types:
field          object
specz_idx       int64
raw_idx         int64
RA            float64
DEC           float64
z_best        float64
is_lbg          int64
mag_u         float64
err_u         float64
snr_u         float64
flag_u          int64
mag_g         float64
err_g         float64
snr_g         float64
flag_g          int64
mag_r         float64
err_r         float64
snr_r         float64
flag_r          int64
mag_i         float64
err_i         float64
snr_i         float64
flag_i          int64
mag_z         float64
err_z         float64
snr_z         float64
flag_z          int64
mag_y         float64
err_y         float64
snr_y         float64
flag_y          int64
u_g           float64
g_r           float64
r_i           float64
i_z           float64
z_y           float64
offset_mag    float64
class_star    float64
obj_type      float64
photo_z       float64
dtype: object
